# Entregable 2: piloto CUDA desde una fuente congelada

Este notebook prepara un entorno Linux x86_64 con Python 3.12.12 aislado y PyTorch 2.8.0 + CUDA 12.6. Está pensado para Colab o un servidor con GPU compatible. La disponibilidad, cuota y costo de GPU dependen del proveedor; este documento no reserva ni contrata recursos.

1. Selecciona una GPU en tu plataforma y ejecuta las celdas en orden.
2. La configuración inicial deja `RUN_PILOT=False`: instala y comprueba el entorno, pero no descarga los CSV ni entrena modelos.
3. Para ejecutar el piloto registrado de **catorce referencias, dos épocas por etapa**, cambia ese valor a `True` y ejecuta las celdas del piloto. No inicia NAS/HPO masivos ni fija el R final.

El código se obtiene de un commit literal de GitHub, sin checkout de datos. Después de verificar CUDA se descargan exclusivamente los dos derivados 2023–2024 fijados por SHA256. Los valores finales de 2025–2026 no se abren.

La instalación descarga paquetes grandes; su tiempo queda fuera del presupuesto de ajustes. Los resultados pertenecen al entorno donde ejecutes el notebook y no se suben automáticamente a GitHub ni a otra aplicación.


In [ ]:
from pathlib import Path
import os, re, sys, json, hashlib, subprocess, platform, shutil, uuid
from datetime import datetime, timezone

# Fuente inmutable del ejecutor y el registro GPU.
SOURCE_COMMIT = "728277517be653be0d0ef10fb677a68adec22bb6"
EXPECTED_GPU_REGISTRATION_SHA256 = "892ad148e1a261d16b178371a8f8c71ea3af352cc4c6ac4fea934df7dc22ba87"
DATA_COMMIT = "0b079b50625a936456a70695532853fc13298f31"
REPOSITORY_URL = "https://github.com/jassan-bit/DeepLearning_202630.git"
RAW_BASE = "https://raw.githubusercontent.com/jassan-bit/DeepLearning_202630/"

RUN_PILOT = False
MODELS = None  # Usa las 14 referencias del registro; ejemplo opcional: ["GRU32", "TimeMixer"]
COHORTS = ["small"]
INCLUDE_LARGEST = False
DEVICE = "cuda:0"
# Puedes elegir un volumen persistente que ya hayas montado en tu plataforma.
WORKSPACE = Path.cwd()/"entregable2_gpu_workspace"
REPO = WORKSPACE/"source"
VENV = WORKSPACE/"venv-py312-cu126"
UV_VERSION = "0.9.5"
MANAGED_PYTHON_VERSION = "3.12.12"
PIP_VERSION = "25.3"
UV_WHEEL_SHA256 = "6507bbbcd788553ec4ad5a96fa19364dc0f58b023e31d79868773559a83ec181"
PIP_WHEEL_SHA256 = "9655943313a94722b7774661c21049070f6bbb0a1516bf02f7c8d5d9201514cd"
DATA_CACHE = WORKSPACE/"development_cache"
OUTPUT_BASE = WORKSPACE/"results"

# Debe establecerse antes de cualquier import/inicialización CUDA de Torch.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
CUDA_ENV = dict(os.environ)
CUDA_ENV.update(CUBLAS_WORKSPACE_CONFIG=":4096:8", PYTHONNOUSERSITE="1",
                OMP_NUM_THREADS="2", MKL_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2")
CUDA_ENV.pop("PYTHONPATH", None)
CUDA_ENV.update(UV_CACHE_DIR=str(WORKSPACE/"uv-cache"),
                UV_PYTHON_INSTALL_DIR=str(WORKSPACE/"managed-python"),
                UV_PYTHON_BIN_DIR=str(WORKSPACE/"managed-python-bin"),
                UV_NATIVE_TLS="true")
HARDWARE_READY = False
GPU_READY = False
DATA_READY = False
PILOT_OUTPUT = None

def run(args, *, cwd=None, capture=False, input_text=None):
    return subprocess.run([str(x) for x in args], cwd=cwd, env=CUDA_ENV, check=True,
                          text=True, input=input_text, capture_output=capture)

def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024*1024), b""):
            h.update(block)
    return h.hexdigest()


## 1. Comprobar la máquina antes de instalar o descargar datos

El kernel de Colab puede usar Python 3.13 o 3.14: no es el intérprete que ejecutará el piloto. Este notebook se detiene si falta Linux x86_64 o una GPU NVIDIA visible. Tener una biblioteca CUDA instalada no equivale a disponer de GPU. La comprobación de kernels CUDA se hará con el entorno aislado, antes de cargar los CSV.


In [ ]:
HARDWARE_READY = False
GPU_READY = False
DATA_READY = False
if not re.fullmatch(r"[0-9a-f]{40}", SOURCE_COMMIT):
    raise RuntimeError("SOURCE_COMMIT debe contener el SHA GPU publicado de 40 caracteres; no usar main.")
if platform.system() != "Linux" or platform.machine() != "x86_64":
    raise RuntimeError("El lock CUDA corresponde a Linux x86_64.")
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No hay GPU NVIDIA visible. Activa GPU en el proveedor antes de continuar.")
probe = run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture=True)
if not probe.stdout.strip():
    raise RuntimeError("nvidia-smi no informó ninguna GPU.")
print(probe.stdout.strip())
WORKSPACE.mkdir(parents=True, exist_ok=True)
HARDWARE_READY = True
print("Python del kernel:", ".".join(map(str, sys.version_info[:3])))


## 2. Descargar sólo el código y sus protocolos

El checkout es parcial: no incluye carpetas `results`, los CSV finales, notebooks anteriores ni diarios de otras ejecuciones. La fuente del programa y la configuración registrada quedan vinculadas al SHA indicado.


In [ ]:
SPARSE_PATHS = [
    "/scripts/", "/tests/", "/requirements-e2-training-gpu.lock.txt",
    "/book_e2/_static/search_protocol_20261109/protocol.json",
    "/experiments/dl_r2_cv_20261104/protocol.json",
    "/experiments/e2_development_pilot_20261109/source_protocol.json",
    "/experiments/e2_gpu_pilot_20261109/",
]
if not REPO.exists():
    REPO.mkdir(parents=True)
    run(["git", "init", REPO])
    run(["git", "-C", REPO, "remote", "add", "origin", REPOSITORY_URL])
    run(["git", "-C", REPO, "config", "core.sparseCheckout", "true"])
    run(["git", "-C", REPO, "config", "remote.origin.promisor", "true"])
    run(["git", "-C", REPO, "config", "remote.origin.partialclonefilter", "blob:none"])
    run(["git", "-C", REPO, "sparse-checkout", "set", "--no-cone", "--stdin"],
        input_text="\n".join(SPARSE_PATHS)+"\n")
    run(["git", "-C", REPO, "fetch", "--filter=blob:none", "--depth=1", "origin", SOURCE_COMMIT])
    run(["git", "-C", REPO, "checkout", "--detach", "FETCH_HEAD"])
if run(["git", "-C", REPO, "rev-parse", "HEAD"], capture=True).stdout.strip() != SOURCE_COMMIT:
    raise RuntimeError("El checkout existente pertenece a otro commit; usa un WORKSPACE nuevo.")
if (REPO/"results").exists():
    raise RuntimeError("Este checkout debe contener sólo código; usa una carpeta nueva sin results.")
print("Fuente verificada:", SOURCE_COMMIT)


## 3. Crear Python 3.12.12 y CUDA en un entorno aislado

El kernel de Colab conserva su versión, aunque sea 3.13 o 3.14. Se instala **uv 0.9.5** en esta carpeta de trabajo, con el hash del wheel oficial, para obtener un CPython **3.12.12** gestionado. Ese Python crea un venv independiente con **pip 25.3**; después se instala el lock CUDA sin modificar el Torch ni los paquetes del kernel.

Se muestran por separado las versiones del kernel y del venv. Si falla una descarga, la creación del intérprete o la comprobación CUDA, `GPU_READY` y `DATA_READY` permanecen en `False`: no se habilita la descarga de datos ni el entrenamiento. Los componentes descargados y la caché quedan dentro de `WORKSPACE`. No se usa apt, conda ni un cambio de kernel.

La instalación necesita red y un driver NVIDIA compatible. La operación CUDA real se comprueba con el Python aislado, antes de cargar los CSV. El tiempo de preparar Python y paquetes está fuera del presupuesto de ajustes.


In [ ]:
GPU_READY = False
DATA_READY = False
if not HARDWARE_READY:
    raise RuntimeError("Comprueba primero Linux y la GPU NVIDIA en la sección 1.")

def bootstrap_python312():
    bootstrap = WORKSPACE/"bootstrap"
    bootstrap.mkdir(parents=True, exist_ok=True)
    uv_home = bootstrap/("uv-"+UV_VERSION)
    uv_binary = uv_home/"bin/uv"
    uv_requirements = bootstrap/"uv.requirements.txt"
    uv_requirements.write_text("uv=="+UV_VERSION+" --hash=sha256:"+UV_WHEEL_SHA256+"\n", encoding="utf-8")
    if not uv_binary.is_file():
        if uv_home.exists():
            raise RuntimeError("El bootstrap uv está incompleto; elige un WORKSPACE nuevo.")
        run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-cache-dir",
             "--no-deps", "--only-binary=:all:", "--require-hashes", "--index-url", "https://pypi.org/simple",
             "--target", uv_home, "-r", uv_requirements])
    uv_version = run([uv_binary, "--version"], capture=True).stdout.strip().split()
    if uv_version[:2] != ["uv", UV_VERSION]:
        raise RuntimeError("La versión local de uv no coincide con la fijada.")
    managed_directory = WORKSPACE/"managed-python"
    run([uv_binary, "python", "install", MANAGED_PYTHON_VERSION,
         "--install-dir", managed_directory, "--no-bin", "--no-config"])
    managed_python = Path(run([uv_binary, "python", "find", MANAGED_PYTHON_VERSION,
                              "--managed-python", "--no-python-downloads", "--no-project", "--no-config"],
                             capture=True).stdout.strip())
    if managed_directory.resolve() not in managed_python.resolve().parents:
        raise RuntimeError("uv seleccionó un Python fuera del directorio gestionado.")
    if not (VENV/"bin/python").is_file():
        run([uv_binary, "venv", "--python", managed_python, "--no-python-downloads", "--no-project",
             "--no-config", VENV])
    python = VENV/"bin/python"
    info = json.loads(run([python, "-c", "import sys,json; print(json.dumps({'version':list(sys.version_info[:3])}))"],
                          capture=True).stdout)
    if info["version"] != [3, 12, 12]:
        raise RuntimeError("El venv no usa CPython 3.12.12; elige una carpeta nueva.")
    pip_requirements = bootstrap/"pip.requirements.txt"
    pip_requirements.write_text("pip=="+PIP_VERSION+" --hash=sha256:"+PIP_WHEEL_SHA256+"\n", encoding="utf-8")
    run([uv_binary, "pip", "install", "--python", python, "--no-deps", "--only-binary=:all:",
         "--require-hashes", "--index-url", "https://pypi.org/simple", "--no-config", "-r", pip_requirements])
    run([python, "-c", "import pip; assert pip.__version__=='25.3'"])
    print("Python del kernel:", ".".join(map(str, sys.version_info[:3])))
    print("Python aislado del piloto:", ".".join(map(str, info["version"])), "con pip", PIP_VERSION)
    return python

GPU_PYTHON = bootstrap_python312()
run([GPU_PYTHON, "-m", "pip", "install", "-r", REPO/"requirements-e2-training-gpu.lock.txt"])
run([GPU_PYTHON, "-m", "pip", "check"])
run([GPU_PYTHON, "-c", "import sys,torch; assert sys.version_info[:3]==(3,12,12); assert torch.__version__=='2.8.0+cu126'; assert torch.version.cuda=='12.6'; print('Python/Torch/CUDA: versiones verificadas')"])
# Sólo la prueba CUDA real habilita las siguientes celdas; no se abren datos aquí.
run([GPU_PYTHON, REPO/"scripts/run_e2_gpu_pilot.py", "--check-environment", "--device", DEVICE], cwd=REPO)
GPU_READY = True


## 4. Preparar los bytes del protocolo y la selección del piloto

El helper histórico comprueba los hashes y conserva una copia del protocolo antes de normalizar el salto de línea usado por los experimentos. Se permite únicamente esa modificación registrada. No se cambian arquitectura, precisión, optimizador ni reglas de parada. Para iTransformer, el helper descarga sus archivos de autor fijados por hash; no descarga pesos preentrenados.

Con `RUN_PILOT=True` esta sección también ejecuta pruebas sintéticas de los 14 modelos antes de cargar datos. Son comprobaciones de forward/backward, sin observaciones de mercado. Sus tiempos no forman parte del piloto de costos.


In [ ]:
def ensure_itransformer_sources():
    import ast
    source = REPO/"scripts/fetch_e2_itransformer_sources.py"
    output = REPO/"work/author_sources/iTransformer"
    def reject_symlinks(path):
        if any(part.is_symlink() for part in (path.absolute(), *path.absolute().parents)):
            raise RuntimeError("El árbol iTransformer no admite enlaces simbólicos.")
    reject_symlinks(source)
    reject_symlinks(output)
    frozen = run(["git", "-C", REPO, "show", SOURCE_COMMIT+":scripts/fetch_e2_itransformer_sources.py"], capture=True).stdout
    if source.read_bytes() != frozen.encode():
        raise RuntimeError("El descargador iTransformer difiere de la fuente congelada.")
    constants = {node.targets[0].id: ast.literal_eval(node.value)
                 for node in ast.parse(frozen).body if isinstance(node, ast.Assign)
                 and len(node.targets) == 1 and isinstance(node.targets[0], ast.Name)
                 and node.targets[0].id == "MANIFEST"}
    manifest, maximum = constants["MANIFEST"], 1024*1024
    if not output.exists():
        run([GPU_PYTHON, source, "--output", "work/author_sources/iTransformer"], cwd=REPO)
    if not output.is_dir():
        raise RuntimeError("El destino iTransformer no es un directorio válido.")
    for path in output.rglob("*"):
        reject_symlinks(path)
        relative = path.relative_to(output).as_posix()
        if path.is_file() and path.suffix in {".py", ".pyc", ".pyo", ".pth", ".so"} and relative not in manifest["files"]:
            if "__pycache__" not in path.relative_to(output).parts or path.suffix != ".pyc":
                raise RuntimeError("Hay código no registrado en el árbol iTransformer.")
    for relative, expected in manifest["files"].items():
        path = output/relative
        reject_symlinks(path)
        if not path.is_file() or path.stat().st_size > maximum or sha256_file(path) != expected:
            raise RuntimeError("Fuente iTransformer ausente o con hash distinto: "+relative)
    receipt = output/"source_receipt.json"
    reject_symlinks(receipt)
    expected_receipt = {"status":"complete", "repository":manifest["repository"], "commit":manifest["commit"],
                        "verified_files":manifest["files"], "weights_downloaded":False, "market_data_read":False}
    if not receipt.is_file() or receipt.stat().st_size > maximum or json.loads(receipt.read_text()) != expected_receipt:
        raise RuntimeError("El recibo iTransformer no coincide con los ocho archivos registrados.")
    print("iTransformer: ocho fuentes verificadas; caché reutilizable.")

run([GPU_PYTHON, REPO/"scripts/prepare_e2_pilot_snapshot.py", "--root", REPO], cwd=REPO)
if RUN_PILOT or MODELS is None or "iTransformer" in MODELS:
    ensure_itransformer_sources()

PROTOCOL_RELATIVE = "book_e2/_static/search_protocol_20261109/protocol.json"
NORMALIZED_PROTOCOL_SHA256 = "e0030c990a4afa15665aec254211b0d9e680e9ac5d92dfb8094e43e965152abe"
changed = run(["git", "-C", REPO, "diff", "--name-only", "HEAD"], capture=True).stdout.splitlines()
if any(name != PROTOCOL_RELATIVE for name in changed):
    raise RuntimeError("Hay cambios en la fuente congelada; usa un checkout nuevo.")
if sha256_file(REPO/PROTOCOL_RELATIVE) != NORMALIZED_PROTOCOL_SHA256:
    raise RuntimeError("El protocolo no coincide con la instantánea registrada.")
untracked = run(["git", "-C", REPO, "ls-files", "--others", "--exclude-standard", "scripts"], capture=True).stdout.splitlines()
if any(name.endswith(".py") for name in untracked):
    raise RuntimeError("Hay módulos Python no registrados en scripts.")
REGISTRATION_RELATIVE = "experiments/e2_gpu_pilot_20261109/registration.json"
REGISTRATION = REPO/REGISTRATION_RELATIVE
registered_bytes = run(["git", "-C", REPO, "show", SOURCE_COMMIT+":"+REGISTRATION_RELATIVE], capture=True).stdout.encode()
if REGISTRATION.read_bytes() != registered_bytes:
    raise RuntimeError("El registro local fue modificado respecto del commit congelado.")
REGISTRATION_SHA256 = hashlib.sha256(registered_bytes).hexdigest()
if REGISTRATION_SHA256 != EXPECTED_GPU_REGISTRATION_SHA256:
    raise RuntimeError("SHA256 del registro GPU distinto del publicado.")
if RUN_PILOT:
    if not GPU_READY:
        raise RuntimeError("Verifica CUDA antes de ejecutar los tests de modelos.")
    smoke_env = dict(CUDA_ENV)
    smoke_env["E2_ITRANSFORMER_SOURCE_ROOT"] = str(REPO/"work/author_sources/iTransformer")
    for test_pattern in ["test_e2_cuda_models.py", "test_e2_cuda_training.py"]:
        subprocess.run([str(GPU_PYTHON), "-m", "unittest", "discover", "-s", "tests", "-p", test_pattern, "-v"], cwd=REPO, env=smoke_env, check=True)
registered_config = json.loads(registered_bytes)
SELECTED_MODELS = registered_config["profile"]["models"] if MODELS is None else list(MODELS)
if not SELECTED_MODELS or any(name not in registered_config["profile"]["models"] for name in SELECTED_MODELS):
    raise RuntimeError("La selección de modelos está fuera del registro.")
print("Piloto seleccionado:", SELECTED_MODELS, COHORTS, "RUN_PILOT=", RUN_PILOT)


## 5. Registrar el lanzamiento y obtener el desarrollo admitido

Con `RUN_PILOT=False` esta celda no descarga datos. Al activarlo, escribe primero un plan local y después verifica los SHA256 de los dos derivados 2023–2024. Se conserva el calendario incompleto original y el loader aplica el corte de cada cohorte; no se interpolan ni se abren archivos de 2025–2026.

El presupuesto global del piloto es de 900 segundos para los ajustes, con límites internos/externos por proceso y memoria definidos en el registro. La instalación, la descarga y otras tareas del proveedor no están cubiertas por ese cronómetro.


In [ ]:
DATA_READY = False
DATA_FILES = {
    "results/eda_development/daily_market_development.csv": "bf6bb5160af924aef6a15c3684aeae932df3017e8639888bb7e91ce787900171",
    "results/eda_development/daily_volatility_development.csv": "20b96374a659eb87dadfa68aa3d6b6b5fb6bea5b10eebe6f2c186ad69861844a",
}
if RUN_PILOT:
    if not GPU_READY:
        raise RuntimeError("CUDA debe verificarse antes de descargar datos.")
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"-"+uuid.uuid4().hex[:8]
    OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
    PILOT_OUTPUT = OUTPUT_BASE/run_id
    launch_plan = {"version":"e2-gpu-notebook-launch-v1", "source_commit":SOURCE_COMMIT,
                   "registration_sha256":REGISTRATION_SHA256, "data_commit":DATA_COMMIT,
                   "models":SELECTED_MODELS, "cohorts":COHORTS, "include_largest":INCLUDE_LARGEST,
                   "device":DEVICE, "data_sha256":DATA_FILES,
                   "scientific_HPO_or_NAS_executed":False, "final_R":None}
    with (OUTPUT_BASE/(run_id+"-notebook-plan.json")).open("x", encoding="utf-8") as stream:
        json.dump(launch_plan, stream, ensure_ascii=False, indent=2, allow_nan=False)
        stream.write("\n")
    # Helper: dos CSV de desarrollo y su protocolo E1, todos fijados por hash.
    run([GPU_PYTHON, REPO/"scripts/fetch_e2_development_sources.py", "--output", DATA_CACHE], cwd=REPO)
    for relative, checksum in DATA_FILES.items():
        if sha256_file(DATA_CACHE/relative) != checksum:
            raise RuntimeError("El derivado no coincide con su hash registrado.")
    DATA_READY = True
    print("Datos de desarrollo verificados; no se cargó el período final.")
else:
    print("Preparación lista. Para ejecutar los perfiles registrados, cambia RUN_PILOT=True y ejecuta desde la sección 4.")


## 6. Ejecutar exclusivamente el piloto registrado

Se utilizan dos épocas fijas y la receta publicada. La opción inicial son las catorce referencias en la cohorte pequeña; no es una comparación final ni una búsqueda de ganadores. Los fallos por tiempo, memoria o incompatibilidad quedan en sus recibos. Cada ejecución usa una carpeta nueva y no sustituye los intentos anteriores.

Para una comprobación opcional de dos referencias, fija `MODELS=["GRU32", "TimeMixer"]` antes del lanzamiento. El valor por defecto `MODELS=None` conserva los catorce modelos del registro. El límite global puede dejar casos sin intentar y ese estado se conserva.


In [ ]:
if RUN_PILOT:
    if not GPU_READY or not DATA_READY or PILOT_OUTPUT is None:
        raise RuntimeError("Ejecuta primero el preflight y la celda de datos/plan.")
    if PILOT_OUTPUT.exists():
        raise RuntimeError("La salida ya existe; crea un lanzamiento nuevo sin borrar sus resultados.")
    command = [GPU_PYTHON, REPO/"scripts/run_e2_gpu_pilot.py",
               "--data-root", DATA_CACHE, "--registration", REGISTRATION,
               "--expected-registration-sha256", REGISTRATION_SHA256,
               "--registration-commit", SOURCE_COMMIT, "--device", DEVICE,
               "--output", PILOT_OUTPUT, "--cohorts", *COHORTS]
    if MODELS is not None:
        command += ["--models", *MODELS]
    if INCLUDE_LARGEST:
        command.append("--include-largest")
    if "iTransformer" in SELECTED_MODELS:
        command += ["--source-root-itransformer", REPO/"work/author_sources/iTransformer"]
    completed = subprocess.run([str(item) for item in command], cwd=REPO, env=CUDA_ENV, text=True)
    print("Código de salida del piloto:", completed.returncode)
    print("Resultados locales:", PILOT_OUTPUT)
    if completed.returncode:
        print("Conserva el intento y revisa su estado: un perfil incompleto no cuenta como entrenamiento completo.")
else:
    print("No se ejecutó entrenamiento: RUN_PILOT=False.")


## 7. Revisar y conservar resultados

Consulta `scientific_report.json`, el informe científico generado por el CLI, junto con sus fallos y costos. Las trazas completas permanecen en el entorno de ejecución: no se suben a GitHub. Descarga el informe científico desde el explorador de archivos de tu plataforma o copia los resultados a un volumen persistente que controles antes de cerrar la sesión.

Un piloto CUDA satisfactorio permite estimar tiempos y memoria en esa GPU. Aún faltan el presupuesto viable, NAS/HPO completos y el piloto de potencia de las recetas ganadoras para fijar R. El período final de evaluación se utiliza después de cerrar esas decisiones. **Este notebook no acredita un rendimiento GPU todavía no medido.**


In [ ]:
if RUN_PILOT and PILOT_OUTPUT is not None:
    report = PILOT_OUTPUT/"scientific_report.json"
    if report.is_file():
        scientific_report = json.loads(report.read_text(encoding="utf-8"))
        print(json.dumps(scientific_report, ensure_ascii=False, indent=2, allow_nan=False))
    else:
        print("No se generó un informe científico; conserva el intento y revisa el error en tu entorno.")
    print("Resultados locales:", PILOT_OUTPUT)
else:
    print("El notebook quedó preparado; no hay resultados de entrenamiento en esta ejecución.")
